# Load neuron mesh and spines with head/neck classification

This notebook demonstrates how to:
1. Load a neuron mesh from an `.obj` file and scale it by 1e-3 (nm → µm)
2. Load the corresponding `.h5` file containing spines with head/neck data
3. Access head-only and neck-only spine meshes
4. Interactively select and zoom on individual spines using k3d

In [1]:
import trimesh
from trimesh import util as triutil
from trimesh.visual.color import ColorVisuals

import morph_spines 
print(morph_spines.__version__)
from morph_spines import load_morphology_with_spines

1.0.0


## Load and scale the neuron mesh

The `.obj` mesh is in nanometers; scale by 1e-3 to convert to micrometers.

In [23]:
mesh_path = "/ssd1/data/microns_v1718/testing/864691134886335738.obj"
neuron_mesh = trimesh.load(mesh_path)
#neuron_mesh.vertices *= 1e-3

print(f"Neuron mesh: {len(neuron_mesh.vertices)} vertices, {len(neuron_mesh.faces)} faces")

Neuron mesh: 1056396 vertices, 2166321 faces


## Load the morphology with spines (head/neck data)

Load the `.h5` file with `load_meshes=True` to preload spine meshes including
head/neck triangle classification.

In [24]:
h5_path = "/ssd1/data/microns_v1718/testing/864691134886335738_skeletonization_8/864691134886335738.h5"
m = load_morphology_with_spines(h5_path, spines_are_centered=True, load_meshes=True)
print(f"Spine count: {m.spines.spine_count}")


HDF5 GROUP:0:warning


Spine count: 1094


## Access spine head and neck meshes

Use `include_head=False` to get only the neck, or `include_neck=False` to get only the head.

In [25]:
spine_idx = 0

full_mesh = m.spines.spine_mesh(spine_idx)
neck_mesh = m.spines.spine_mesh(spine_idx, include_head=False)
head_mesh = m.spines.spine_mesh(spine_idx, include_neck=False)

print(f"Full spine: {len(full_mesh.faces)} faces")
print(f"Neck only:  {len(neck_mesh.faces)} faces")
print(f"Head only:  {len(head_mesh.faces)} faces")

Full spine: 3141 faces
Neck only:  1255 faces
Head only:  1886 faces


## Interactive spine viewer (k3d)

Use the dropdown to select a spine. The viewer shows the spine head (red), neck (green),
and a cropped region of the neuron mesh (gray) for context.

In [26]:
import numpy as np

# Check if neuron mesh and spines are in the same coordinate space
print("Neuron mesh:")
print(f"  min: {neuron_mesh.vertices.min(axis=0)}")
print(f"  max: {neuron_mesh.vertices.max(axis=0)}")
print(f"  extent: {neuron_mesh.vertices.max(axis=0) - neuron_mesh.vertices.min(axis=0)}")
print()
spine0 = m.spines.spine_mesh(0)
print("Spine 0:")
print(f"  min: {spine0.vertices.min(axis=0)}")
print(f"  max: {spine0.vertices.max(axis=0)}")
print(f"  centroid: {spine0.centroid}")
print()
# Check if spine centroid falls within neuron mesh bounds
nm_min = neuron_mesh.vertices.min(axis=0)
nm_max = neuron_mesh.vertices.max(axis=0)
c = spine0.centroid
inside = np.all(c >= nm_min) and np.all(c <= nm_max)
print(f"Spine 0 centroid inside neuron bbox: {inside}")


Neuron mesh:
  min: [605.28302  348.075012 755.433044]
  max: [ 888.773621  762.867065 1113.815552]
  extent: [283.490601 414.792053 358.382508]

Spine 0:
  min: [691.37378529 589.64490162 869.47889521]
  max: [692.70280746 591.28070665 871.5817424 ]
  centroid: [691.9495108  590.18214919 870.15241948]

Spine 0 centroid inside neuron bbox: True


In [ ]:
import numpy as np
import k3d
import ipywidgets as widgets
from IPython.display import display

# Neuron mesh vertices as point cloud
neuron_pts = neuron_mesh.vertices.astype(np.float32)
print(f'Neuron point cloud: {len(neuron_pts)} points')

# Create K3D plot
plot = k3d.plot(grid_visible=False, background_color=0xffffff)

# Draw the neuron mesh as point cloud (constant screen-space size)
plot += k3d.points(
    neuron_pts,
    point_size=0.05,
    color=0xaaaaaa,
    opacity=0.5,
    shader='gaussian',
)

# State: track spine mesh objects for clean removal
spine_meshes = []

def display_spine(spine_id):
    """Display a spine (head/neck) and focus the camera on it."""
    global spine_meshes, plot

    # Remove previous spine meshes
    for obj in spine_meshes:
        try:
            plot -= obj
        except Exception:
            pass
    spine_meshes.clear()

    neck = m.spines.spine_mesh(spine_id, include_head=False)
    head = m.spines.spine_mesh(spine_id, include_neck=False)

    if len(neck.faces) > 0:
        obj = k3d.mesh(
            neck.vertices.astype(np.float32),
            neck.faces.astype(np.uint32),
            color=0x64c864,
            flat_shading=True,
        )
        plot += obj
        spine_meshes.append(obj)

    if len(head.faces) > 0:
        obj = k3d.mesh(
            head.vertices.astype(np.float32),
            head.faces.astype(np.uint32),
            color=0xff6464,
            flat_shading=True,
        )
        plot += obj
        spine_meshes.append(obj)

    # Focus camera on the spine
    full = m.spines.spine_mesh(spine_id)
    center = full.centroid.astype(np.float32)
    radius = float(np.linalg.norm(full.vertices - center, axis=1).max())
    camera_pos = center + np.array([0, 0, 4.0 * radius], dtype=np.float32)
    plot.camera_auto_fit = False
    plot.camera = camera_pos.tolist() + center.tolist() + [0, 1, 0]

# Build list of valid spine indices (those with non-empty meshes)
valid_spines = []
for i in range(m.spines.spine_count):
    try:
        mesh = m.spines.spine_mesh(i)
        if mesh is not None and len(mesh.faces) > 0:
            valid_spines.append(i)
    except Exception:
        pass
print(f'Valid spines: {len(valid_spines)} / {m.spines.spine_count}')

# Dropdown for spine selection
dropdown = widgets.Dropdown(
    options=valid_spines,
    value=valid_spines[0],
    description='Spine #:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px'),
)

def on_change(change):
    if change.get('name') == 'value':
        display_spine(change['new'])

dropdown.observe(on_change, names='value')

# Camera reset button
reset_btn = widgets.Button(description='Reset Camera', button_style='primary')

def reset_camera(_=None):
    plot.camera_reset()
    plot.camera_auto_fit = True

reset_btn.on_click(reset_camera)

# Layout and display
controls = widgets.HBox([dropdown, reset_btn])
display(widgets.VBox([plot, controls]))
display_spine(valid_spines[0])


Neuron point cloud: 1056396 points
Valid spines: 1094 / 1094
